In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import pint
import pint_xarray
from imagematerials.util import dataset_to_array
from imagematerials.util import import_from_netcdf, export_to_netcdf

from constants import vehicle_type_mapping, target_regions, region_mapping, target_variables_slow, target_variables_narrow
# Import Preprocesssed data
vehicle_pre = import_from_netcdf("../../examples/prep_vema2.nc")
# ['battery_materials', 'battery_shares', 'weight_boats', 'material_fractions', 'vehicle_weights', 
# 'battery_weights', 'stocks', 'shares', 'maintenance_material_fractions', 'lifetimes']



In [2]:
# CREATE XARRAY to store data
all_variables = list(target_variables_narrow.keys()) + list(target_variables_slow.keys())

vehicle_targets = xr.DataArray(
            0.0,
            dims=("year", "region", "variable"),
            coords={"year": [2020,2060],
                    "region": target_regions,
                    "variable": all_variables})

In [3]:
image_output = pd.read_csv("IMAGE scenario explorer variables.csv", delimiter=";", index_col=0, usecols=range(8))
image_output["unit"] = image_output["unit"].astype(str).str.strip() 
image_output["unit"].unique()

array(['Mt/yr', 'Mt', 'km', 'Mt CO2/yr', 'Mt CO2-equiv/yr', 'bn m2',
       'billion tkm/yr', 'billion pkm/yr', 'billion USD_2010/yr',
       'million', 'US$2010/GJ', 'EJ/yr', 'm^2', 'thousand'], dtype=object)

In [ ]:
# Import model output data
image_output = pd.read_csv("IMAGE scenario explorer variables.csv", delimiter=";", index_col=0, usecols=range(8))

# TRANSFORM model output to xarray

# Define a unit registry
#unit_registry = pint_xarray.unit_registry

# Assuming image_output is a Pandas DataFrame with 'variable' and 'unit' columns
#unit_mapping = dict(zip(image_output["variable"], image_output["unit"]))

image_output = image_output.drop(columns=["model", "scenario","unit"])

# Step 2: Convert DataFrame into xarray Dataset
xr_image_output = image_output.set_index(["variable", "region", "year"]).to_xarray()

xr_image_output = xr_image_output.sel(year=[2020, 2060])

xr_image_output = xr_image_output.to_dataarray("var2").drop_vars("var2").squeeze()

#xr_image_output = xr_image_output.pint.quantify(unit_mapping)

In [5]:
image_output

,region,variable,year,value
0,World,Material Inflow|Aggregates|Infrastructure|Mobi...,2005,0.000000
1,World,Material Inflow|Aggregates|Infrastructure|Mobi...,2010,0.000000
2,World,Material Inflow|Aggregates|Infrastructure|Mobi...,2015,0.000000
3,World,Material Inflow|Aggregates|Infrastructure|Mobi...,2020,0.000000
4,World,Material Inflow|Aggregates|Infrastructure|Mobi...,2025,0.000000
...,...,...,...,...
70297,World,Value Added|Services,2060,0.219616
70298,World,Value Added|Services,2070,0.268755
70299,World,Value Added|Services,2080,0.320978
70300,World,Value Added|Services,2090,0.376271


In [5]:
# REGIONAL AGGREGATION

# Convert the region names in xr_image_output to match the target regions
xr_regions = xr_image_output.coords["region"].values  # Extract current region names

# Create a mapping of only existing regions
region_mapping_filtered = {key: val for key, val in region_mapping.items() if key in xr_regions}

# Map the dataset’s regions to the target regions
xr_image_output_filtered = xr_image_output.sel(region=list(region_mapping_filtered.keys()))
xr_image_output_filtered = xr_image_output_filtered.assign_coords(region=[region_mapping_filtered[r] for r in xr_image_output_filtered["region"].values])

# Aggregate by summing over the mapped regions
xr_image_output_aggregated = xr_image_output_filtered.groupby("region").sum()

xr_image_output_aggregated.coords["region"] = xr_image_output_aggregated.coords["region"].reindex_like(vehicle_targets.coords["region"] )


In [6]:
# EXTRACT SERVICE DEMAND
pop = xr_image_output_aggregated.sel(variable="Population")  # Extract Population data
energy_service = xr_image_output_aggregated.sel(variable=[v for v in xr_image_output_aggregated["variable"].values if "Energy Service|Transportation|" in v or "Product Stock|Transportation|" in v])

energy_service_per_capita = energy_service / pop

vehicle_targets.loc[dict(variable="Total freight. travel demand")] = energy_service_per_capita.sel(variable="Energy Service|Transportation|Freight")
vehicle_targets.loc[dict(variable="Total pass. travel demand")] = energy_service_per_capita.sel(variable="Energy Service|Transportation|Passenger")


In [7]:
# WEIGHTS CALCULATION
vehicle_weights = vehicle_pre["vehicle_weights"]

# Convert the region names in xr_image_output to match the target regions
xr_type = vehicle_weights.coords["Type"].values  # Extract current region names

# Create a mapping of only existing regions
vehicle_type_mapping_filtered = {key: val for key, val in vehicle_type_mapping.items() if key in xr_type}

# Map the dataset’s regions to the target regions
vehicle_weights_filtered = vehicle_weights.sel(Type=list(vehicle_type_mapping_filtered.keys()))
vehicle_weights_filtered = vehicle_weights_filtered.assign_coords(Type=[vehicle_type_mapping_filtered[r] for r in vehicle_weights_filtered["Type"].values])

# Select only the years 2020 and 2060
vehicle_weights_filtered = vehicle_weights_filtered.sel(Cohort=[2020, 2060])

# Exclude zeros by applying a mask (only non-zero values will be considered in the mean)
vehicle_weights_filtered_no_zeros = vehicle_weights_filtered.where(vehicle_weights_filtered != 0)

# Aggregate by summing over the mapped regions, excluding zeros
vehicle_weights_aggregated = vehicle_weights_filtered_no_zeros.groupby("Type").mean()




In [8]:
vehicle_targets.name = "values"

# Select the variables you're interested in from vehicle_targets
selected_variables = ["Total pass. travel demand", "Total freight. travel demand"]

# Extract data for the selected variables
selected_data = vehicle_targets.sel(variable=selected_variables)

# Convert to pandas DataFrame
df = selected_data.to_dataframe().reset_index()

# Pivot the DataFrame to have 'year' as columns
df_pivot = df.pivot_table(index=["region", "variable"], columns="year", values="values")


df_pivot.to_csv("travel_demand.csv")
# Display the DataFrame


In [9]:
# Iterate over the target variables and assign weights to vehicle_targets
for target_var in target_variables_slow:
    if target_var in vehicle_weights_aggregated.coords["Type"].values:
        # Extract the aggregated weight for the corresponding target variable
        weight = vehicle_weights_aggregated.sel(Type=target_var).values
        print(weight)
        
        # Create an array of the same weight for all regions
        broadcasted_weights = np.full(len(vehicle_targets.coords["region"]), weight)
        
        # Assign this array of broadcasted weights to the variable in vehicle_targets
        vehicle_targets.loc[:, target_var, :] = broadcasted_weights


[1500.    1372.526]


ValueError: could not broadcast input array from shape (2,) into shape (13,)